# HUẤN LUYỆN & ĐÁNH GIÁ TỐI ƯU CHO M3 (ASTNN) & M4 (CPG-GNN) TRÊN BIGCLONEBENCH JAVA

Notebook này triển khai phiên bản huấn luyện cải tiến nhằm khắc phục hiện tượng mất hội tụ và sụp đổ nhãn của M3 và M4:
1. **M3 (ASTNN):** Tree-based Neural Network với learning rate tối ưu (3e-4), max_grad_norm=1.0, patience=3 và 8 epochs.
2. **M4 (CPG-GNN):** GAT Neural Network trên Joern Code Property Graph với learning rate 3e-4, patience=3 và 8 epochs.

### Dữ liệu & Đặc trưng:
- **Dataset liên kết:** `kichirou/semanticcodeclonedetection-dataset-m4` chứa AST cache và Joern CPG export.
- **Splits:** 30.000 cặp Train, 3.000 cặp Val, 5.000 cặp Test cân bằng 50:50.
- **Tự động lưu checkpoint & so sánh tổng hợp 4 mô hình** (M1, M2, M3, M4) vào `/kaggle/working/`.

---

In [ ]:
# 1. THIẾT LẬP MÔI TRƯỜNG & HÀM BỔ TRỢ STREAM LOG
import base64, io, json, os, shutil, subprocess, sys, tarfile, time, yaml
from pathlib import Path

def sh(cmd, **kw):
    print(f"\n$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    log_file = Path("/kaggle/working/live_progress.log")
    with log_file.open("a", encoding="utf-8") as lf:
        for line in iter(proc.stdout.readline, ''):
            print(line, end='', flush=True)
            lf.write(line)
            lf.flush()
    proc.stdout.close()
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Lệnh thất bại (rc={rc}): {cmd}")
    return ""

print("Python:", sys.version.split()[0], flush=True)

In [ ]:
# 2. GIẢI NÉN MÃ NGUỒN DỰ ÁN MỚI NHẤT (SELF-CONTAINED)
WORK_DIR = Path("/kaggle/working/project")
WORK_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_B64 = "..."

print(f"Giải nén mã nguồn ({len(BUNDLE_B64)/1024:.1f} KB) vào {WORK_DIR}...")
buf = io.BytesIO(base64.b64decode(BUNDLE_B64))
with tarfile.open(fileobj=buf, mode="r:gz") as tar:
    tar.extractall(WORK_DIR)

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
sh("ls -la configs/")

In [ ]:
# 3. CÀI ĐẶT THƯ VIỆN & XÁC THỰC GPU
sh("pip install -q 'tree-sitter>=0.23.0' 'tree-sitter-java>=0.23.5' 'tree-sitter-cpp>=0.23.0' 'datasets>=2.14.0' 'transformers>=4.30.0' 'torch_geometric' 'pyarrow' 'tqdm' 'pyyaml' 'scikit-learn'")

import torch
assert torch.cuda.is_available(), "Cần bật GPU trong Settings -> Accelerator -> GPU (T4/P100)!"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
print(f"GPU SẴN SÀNG: {gpu_name} | VRAM: {vram_gb} GB")
sh("nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv")

In [ ]:
# 4. LIÊN KẾT ĐẶC TRƯNG TỪ DATASET (AST CACHE & CPG EXPORT)
import numpy as np
ds_candidates = [
    Path("/kaggle/input/datasets/kichirou/semanticcodeclonedetection-dataset-m4"),
    Path("/kaggle/input/semanticcodeclonedetection-dataset-m4"),
]
ds_root = None
for d in ds_candidates:
    if d.exists():
        ds_root = d
        break

print(f"Thư mục dataset phát hiện: {ds_root}")
if ds_root:
    sh(f"ls -la {ds_root}/")

# 1. Đóng gói AST cache cho M3 nếu có thư mục .npy rời
os.makedirs("artifacts/bcb_features", exist_ok=True)
ast_target = Path("artifacts/bcb_features/bcb_ast_cache.npz")
ast_src = ds_root / "bcb_ast_cache" if ds_root else None
if ast_src and ast_src.exists():
    print(f"Phát hiện thư mục AST cache tại: {ast_src}")
    npy_files = list(ast_src.glob("*.npy"))
    if npy_files:
        print(f"Tìm thấy {len(npy_files)} file .npy, tiến hành đóng gói {ast_target.name}...")
        arrays = {f.stem: np.load(f, allow_pickle=True) for f in npy_files}
        np.savez_compressed(ast_target, **arrays)
        print(f"Đã tạo thành công {ast_target} ({ast_target.stat().st_size / 1024:.1f} KB)!")
elif (ds_root / "bcb_ast_cache.npz").exists() if ds_root else False:
    sh(f"cp {ds_root}/bcb_ast_cache.npz {ast_target}")
    print(f"Đã sao chép {ast_target} từ dataset!")
else:
    print("Thông báo: Chưa có pre-computed AST cache, sẽ biên dịch online tự động.")

# 2. Liên kết CPG Graphs cho M4
os.makedirs("artifacts", exist_ok=True)
cpg_src = ds_root / "bcb_cpg_export" if ds_root else None
if cpg_src and cpg_src.exists():
    print(f"Phát hiện thư mục CPG export tại: {cpg_src}")
    inner = cpg_src / "cpg_export"
    dest = Path("artifacts/cpg_export")
    if inner.exists():
        sh(f"cp -r {inner} artifacts/")
    else:
        sh(f"cp -r {cpg_src} artifacts/cpg_export")
    cpg_count = len(list(dest.glob("*.json")))
    print(f"Đã liên kết {cpg_count:,} đồ thị CPG JSON vào {dest}!")
elif (ds_root / "bcb_cpg_export.tar.gz").exists() if ds_root else False:
    sh(f"tar -xzf {ds_root}/bcb_cpg_export.tar.gz -C artifacts/")
    print("Đã giải nén bcb_cpg_export.tar.gz vào artifacts/!")
else:
    print("Cảnh báo: Không tìm thấy tài nguyên CPG export!")

In [ ]:
# 5. TẢI CÁC SPLIT DỮ LIỆU ĐỂ HUẤN LUYỆN (TRAIN 30.000, VAL 3.000, TEST 5.000 CÂN BẰNG 50:50)
data_raw = Path("data/raw/bigclonebench")
data_raw.mkdir(parents=True, exist_ok=True)
print("Tải các phân tập dữ liệu BigCloneBench (Train: 30k, Val: 3k, Test: 5k) cân bằng 50:50...")
sh("python scripts/download_bcb_subsets.py --splits-custom --train-samples 30000 --val-samples 3000 --test-samples 5000")
sh("ls -lh data/raw/bigclonebench/")
assert (data_raw / 'bcb_train.parquet').exists() and (data_raw / 'bcb_train.parquet').stat().st_size > 0, 'bcb_train.parquet phai ton tai va co du lieu!'

In [ ]:
# 6. HUẤN LUYỆN & ĐÁNH GIÁ M3 (ASTNN)
print("=" * 70)
print(">>> [1/2] BẮT ĐẦU HUẤN LUYỆN M3 (ASTNN) - TREE-BASED NEURAL NETWORK...")
print("=" * 70)
try:
    sh("python -u -m src.experiments.bcb_runner --config configs/exp004_astnn_bcb.yaml --mode preflight --device cuda")
    sh("python -u -m src.experiments.bcb_runner --config configs/exp004_astnn_bcb.yaml --mode diagnostic --device cuda")
    sh("python -u -m src.experiments.bcb_runner --config configs/exp004_astnn_bcb.yaml --mode train --device cuda --time-budget-s 18000")
    sh("python -u -m src.experiments.bcb_runner --config configs/exp004_astnn_bcb.yaml --mode evaluate --device cuda")
    m3_res = Path("results/exp_bcb_004")
    m3_out = Path("/kaggle/working/m3_results")
    m3_out.mkdir(parents=True, exist_ok=True)
    if m3_res.exists():
        sh(f"cp -r {m3_res}/* {m3_out}/")
        sh(f"cp {m3_out}/test_metrics.json /kaggle/working/m3_test_metrics.json 2>/dev/null || true")
    print(">>> [1/2] M3 (ASTNN) HOÀN TẤT THÀNH CÔNG!")
except Exception as e:
    print(f">>> [1/2] LỖI TẠI M3 (ASTNN): {e}")
finally:
    import torch; torch.cuda.empty_cache()
    print("Đã giải phóng GPU VRAM sau khi chạy M3.")

In [ ]:
# 7. HUẤN LUYỆN & ĐÁNH GIÁ M4 (CPG-GNN)
print("=" * 70)
print(">>> [2/2] BẮT ĐẦU HUẤN LUYỆN M4 (CPG-GNN) - GRAPH NEURAL NETWORK ON JOERN CPG...")
print("=" * 70)
try:
    sh("python -u -m src.experiments.bcb_runner --config configs/exp005_cpg_gnn_bcb.yaml --mode preflight --device cuda")
    sh("python -u -m src.experiments.bcb_runner --config configs/exp005_cpg_gnn_bcb.yaml --mode diagnostic --device cuda")
    sh("python -u -m src.experiments.bcb_runner --config configs/exp005_cpg_gnn_bcb.yaml --mode train --device cuda --time-budget-s 18000")
    sh("python -u -m src.experiments.bcb_runner --config configs/exp005_cpg_gnn_bcb.yaml --mode evaluate --device cuda")
    m4_res = Path("results/exp_bcb_005")
    m4_out = Path("/kaggle/working/m4_results")
    m4_out.mkdir(parents=True, exist_ok=True)
    if m4_res.exists():
        sh(f"cp -r {m4_res}/* {m4_out}/")
        sh(f"cp {m4_out}/test_metrics.json /kaggle/working/m4_test_metrics.json 2>/dev/null || true")
    print(">>> [2/2] M4 (CPG-GNN) HOÀN TẤT THÀNH CÔNG!")
except Exception as e:
    print(f">>> [2/2] LỖI TẠI M4 (CPG-GNN): {e}")
finally:
    import torch; torch.cuda.empty_cache()
    print("Đã giải phóng GPU VRAM sau khi chạy M4.")

In [ ]:
# 8. TỔNG HỢP BẢNG SO SÁNH 4 MÔ HÌNH VÀ ĐÓNG GÓI KẾT QUẢ
print("=" * 80)
print("TỔNG HỢP BÁO CÁO TOÀN DIỆN CẢ 4 MÔ HÌNH TRÊN BIGCLONEBENCH...")
print("=" * 80)

comparison = {
    'M1': {'threshold': 0.88, 'precision': 0.8712, 'recall': 0.9353, 'f1': 0.9021, 'accuracy': 0.9724, 'roc_auc': 0.9822},
    'M2': {'threshold': 0.94, 'precision': 0.9239, 'recall': 0.9103, 'f1': 0.9170, 'accuracy': 0.9776, 'roc_auc': 0.9768},
}

saved_summary = WORK_DIR / "results/all_models_summary.json"
if saved_summary.exists():
    try:
        comparison.update(json.loads(saved_summary.read_text(encoding='utf-8')))
    except Exception:
        pass

for mid, mp in [('M3', '/kaggle/working/m3_results/test_metrics.json'), ('M4', '/kaggle/working/m4_results/test_metrics.json')]:
    p = Path(mp)
    if p.exists():
        try:
            comparison[mid] = json.loads(p.read_text(encoding='utf-8'))
        except Exception:
            pass

models_display = [
    ('M1', 'CodeBERT (Seq)'),
    ('M2', 'GraphCodeBERT (DFG)'),
    ('M3', 'ASTNN (Tree)'),
    ('M4', 'CPG-GNN (Joern CPG)'),
]

print("\n" + "-" * 85)
print(f"{'Model':<6} | {'Phương pháp':<22} | {'F1-Score':<10} | {'Precision':<10} | {'Recall':<10} | {'Accuracy':<10}")
print("-" * 85)

for mid, mname in models_display:
    if mid in comparison:
        data = comparison[mid]
        f1 = data.get('f1', 0.0)
        prec = data.get('precision', 0.0)
        rec = data.get('recall', 0.0)
        acc = data.get('accuracy', 0.0)
        print(f"{mid:<6} | {mname:<22} | {f1:<10.4f} | {prec:<10.4f} | {rec:<10.4f} | {acc:<10.4f}")
    else:
        print(f"{mid:<6} | {mname:<22} | {'CHƯA CÓ':<10} | {'-':<10} | {'-':<10} | {'-':<10}")
print("-" * 85)

summary_path = Path("/kaggle/working/all_models_summary.json")
summary_path.write_text(json.dumps(comparison, indent=2, ensure_ascii=False), encoding='utf-8')

os.makedirs('results', exist_ok=True)
archive_path = Path('/kaggle/working/bcb_m3_m4_results.tar.gz')
print(f'Đang nén toàn bộ kết quả vào {archive_path}...')
sh(f'tar -czf {archive_path} results/ /kaggle/working/*_results /kaggle/working/*.json 2>/dev/null || true')
sh('ls -lh /kaggle/working/')

print("\n======================================================================")
print("HOÀN TẤT TOÀN BỘ TIẾN TRÌNH HUẤN LUYỆN & ĐÁNH GIÁ TRÊN BIGCLONEBENCH!")
print("======================================================================")